# OTDR candidate-level 4-class model training

Это честная постановка для end-to-end event detection.

1. Кандидаты были созданы только из CSV.
2. JSON использовался после генерации candidates только для назначения labels.
3. Файлы train и validation разделены.
4. Модель обучается на `bend` / `connector` / `break` / `background`.
5. Модель выбирается по event-only macro F1 на validation (`bend`, `connector`, `break`).
6. Затем на validation подбирается confidence threshold для последующего test inference.

In [ ]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             classification_report, confusion_matrix, ConfusionMatrixDisplay)

RANDOM_STATE = 42
EVENT_CLASSES = ['bend', 'connector', 'break']
ALL_CLASSES = EVENT_CLASSES + ['background']

FEATURE_COLS = [
    'm_norm', 'db_at_event', 'local_mean_db', 'local_std_db',
    'pre_mean_db', 'post_mean_db', 'loss_dB', 'peak_above_bg_dB',
    'pre_slope_dB_per_km', 'post_slope_dB_per_km', 'slope_change_dB_per_km',
    'derivative_at_event_dB_per_km', 'max_pre_derivative', 'min_post_derivative',
    'pre_std_db', 'post_std_db', 'post_to_pre_std_ratio',
    'peak_width_m', 'local_range_db'
]

DATA_DIR = Path.cwd() / 'candidate_event_model_data_balanced'
TRAIN_CSV = DATA_DIR / 'train_candidates_balanced.csv'
VAL_CSV = DATA_DIR / 'val_candidates_balanced.csv'
OUT_DIR = DATA_DIR / 'model_results'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('Train CSV:', TRAIN_CSV)
print('Validation CSV:', VAL_CSV)
print('Output:', OUT_DIR)

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)

missing = [c for c in FEATURE_COLS if c not in train_df.columns or c not in val_df.columns]
if missing:
    raise ValueError(f'Missing feature columns: {missing}')

train_df = train_df[train_df['label'].isin(ALL_CLASSES)].dropna(subset=FEATURE_COLS).copy()
val_df = val_df[val_df['label'].isin(ALL_CLASSES)].dropna(subset=FEATURE_COLS).copy()

X_train = train_df[FEATURE_COLS]
y_train = train_df['label']
X_val = val_df[FEATURE_COLS]
y_val = val_df['label']

print('TRAIN class counts:')
display(y_train.value_counts().reindex(ALL_CLASSES, fill_value=0).to_frame('count'))
print('VALIDATION class counts:')
display(y_val.value_counts().reindex(ALL_CLASSES, fill_value=0).to_frame('count'))

assert set(train_df['file']).isdisjoint(set(val_df['file'])), 'Leakage: same files found in train and validation!'
print('OK: train/validation files are disjoint.')

In [ ]:
models = {
    'DecisionTree': DecisionTreeClassifier(
        max_depth=12, min_samples_leaf=4, class_weight='balanced', random_state=RANDOM_STATE
    ),
    'RandomForest': RandomForestClassifier(
        n_estimators=500, max_depth=16, min_samples_leaf=2,
        class_weight='balanced_subsample', random_state=RANDOM_STATE, n_jobs=-1
    ),
    'ExtraTrees': ExtraTreesClassifier(
        n_estimators=500, max_depth=18, min_samples_leaf=2,
        class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1
    ),
    'GradientBoosting': GradientBoostingClassifier(
        n_estimators=300, learning_rate=0.04, max_depth=3, random_state=RANDOM_STATE
    ),
    'LogisticRegression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=4000, class_weight='balanced', random_state=RANDOM_STATE))
    ]),
    'SVM_RBF': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', SVC(kernel='rbf', C=2.0, gamma='scale', class_weight='balanced',
                    probability=True, random_state=RANDOM_STATE))
    ]),
}

min_class_count = int(y_train.value_counts().min())
n_folds = min(5, min_class_count)
cv = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=RANDOM_STATE)
print('CV folds:', n_folds)

In [ ]:
results = []
fitted_models = {}
val_predictions = {}
val_probabilities = {}

for name, model in models.items():
    cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='f1_macro', n_jobs=-1)
    fitted = clone(model).fit(X_train, y_train)
    pred = fitted.predict(X_val)
    proba = fitted.predict_proba(X_val)

    fitted_models[name] = fitted
    val_predictions[name] = pred
    val_probabilities[name] = proba

    _, _, f1_all, _ = precision_recall_fscore_support(
        y_val, pred, labels=ALL_CLASSES, average='macro', zero_division=0
    )
    p_event, r_event, f1_event, _ = precision_recall_fscore_support(
        y_val, pred, labels=EVENT_CLASSES, average='macro', zero_division=0
    )

    p_cls, r_cls, f_cls, support_cls = precision_recall_fscore_support(
        y_val, pred, labels=ALL_CLASSES, zero_division=0
    )
    row = {
        'model': name,
        'cv_f1_macro_all_mean': cv_scores.mean(),
        'cv_f1_macro_all_std': cv_scores.std(),
        'val_accuracy': accuracy_score(y_val, pred),
        'val_f1_macro_all': f1_all,
        'val_precision_macro_events': p_event,
        'val_recall_macro_events': r_event,
        'val_f1_macro_events': f1_event,
    }
    for cls, p, r, f, s in zip(ALL_CLASSES, p_cls, r_cls, f_cls, support_cls):
        row[f'{cls}_precision'] = p
        row[f'{cls}_recall'] = r
        row[f'{cls}_f1'] = f
        row[f'{cls}_support'] = s
    results.append(row)

comparison = pd.DataFrame(results).sort_values('val_f1_macro_events', ascending=False).reset_index(drop=True)
comparison.round(4)

In [ ]:
comparison.to_csv(OUT_DIR / 'model_comparison_candidate_level.csv', index=False)

for name in comparison['model']:
    pred = val_predictions[name]
    print('=' * 88)
    print(name)
    print(classification_report(y_val, pred, labels=ALL_CLASSES, digits=3, zero_division=0))
    fig, ax = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay(
        confusion_matrix(y_val, pred, labels=ALL_CLASSES), display_labels=ALL_CLASSES
    ).plot(ax=ax, cmap='Blues', values_format='d', colorbar=False)
    ax.set_title(f'{name}: candidate-level validation')
    plt.tight_layout()
    plt.show()

## Confidence threshold tuning

Модель уже выбрана только по validation. Теперь подбираем порог confidence тоже только на validation.

Правило: candidate остаётся event только если:
- predicted class не `background`;
- probability выбранного event class >= threshold.

Метрика ниже считает classification-level event precision/recall на candidate rows. Для финального test это не заменяет NMS и event matching, но помогает выбрать консервативный порог до использования test dataset.

In [ ]:
best_name = comparison.iloc[0]['model']
best_model = fitted_models[best_name]
proba = val_probabilities[best_name]
model_classes = list(best_model.classes_) if hasattr(best_model, 'classes_') else list(best_model.named_steps['clf'].classes_)
pred_raw = np.asarray(val_predictions[best_name]).astype(str)

threshold_rows = []
for threshold in np.arange(0.25, 0.96, 0.05):
    pred_thresholded = pred_raw.copy()
    for i, label in enumerate(pred_raw):
        if label in EVENT_CLASSES:
            p = proba[i, model_classes.index(label)]
            if p < threshold:
                pred_thresholded[i] = 'background'

    # Treat each candidate event prediction against candidate GT label.
    tp = sum((y_val.values == pred_thresholded) & np.isin(y_val.values, EVENT_CLASSES))
    fp = sum((pred_thresholded != 'background') & (pred_thresholded != y_val.values))
    fn = sum((y_val.values != 'background') & (pred_thresholded != y_val.values))
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    threshold_rows.append({'threshold': round(float(threshold), 2), 'TP': tp, 'FP': fp, 'FN': fn,
                           'event_precision': precision, 'event_recall': recall, 'event_f1': f1})

threshold_table = pd.DataFrame(threshold_rows).sort_values('event_f1', ascending=False).reset_index(drop=True)
threshold_table.to_csv(OUT_DIR / 'confidence_threshold_tuning.csv', index=False)
threshold_table.round(4)

In [ ]:
best_threshold = float(threshold_table.iloc[0]['threshold'])

plt.figure(figsize=(8, 4))
plt.plot(threshold_table.sort_values('threshold')['threshold'],
         threshold_table.sort_values('threshold')['event_f1'], marker='o', label='F1')
plt.plot(threshold_table.sort_values('threshold')['threshold'],
         threshold_table.sort_values('threshold')['event_precision'], marker='o', label='precision')
plt.plot(threshold_table.sort_values('threshold')['threshold'],
         threshold_table.sort_values('threshold')['event_recall'], marker='o', label='recall')
plt.axvline(best_threshold, color='black', linestyle='--', label=f'best={best_threshold:.2f}')
plt.xlabel('event confidence threshold')
plt.ylabel('score')
plt.title(f'{best_name}: validation threshold tuning')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

print('Selected threshold:', best_threshold)

In [ ]:
bundle = {
    'model': best_model,
    'model_name': best_name,
    'features': FEATURE_COLS,
    'classes': ALL_CLASSES,
    'event_classes': EVENT_CLASSES,
    'recommended_confidence_threshold': best_threshold,
    'selection_metric': 'candidate-level validation event-only macro F1',
    'best_val_f1_macro_events': float(comparison.iloc[0]['val_f1_macro_events']),
    'best_val_f1_macro_all': float(comparison.iloc[0]['val_f1_macro_all']),
    'train_rows': len(train_df),
    'val_rows': len(val_df),
    'note': 'Candidates were generated from CSV only. JSON used after candidate generation only for labels.'
}
model_path = OUT_DIR / 'best_candidate_level_event_model.joblib'
joblib.dump(bundle, model_path)

print('Best model:', best_name)
print('Validation event-only macro F1:', f"{comparison.iloc[0]['val_f1_macro_events']:.4f}")
print('Recommended confidence threshold:', best_threshold)
print('Saved:', model_path)

In [ ]:
if best_name in {'DecisionTree', 'RandomForest', 'ExtraTrees', 'GradientBoosting'}:
    importance = pd.Series(best_model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)
    display(importance.to_frame('importance'))
    fig, ax = plt.subplots(figsize=(9, 6))
    importance.sort_values().plot.barh(ax=ax, color='#1565C0')
    ax.set_title(f'{best_name}: candidate-level feature importances')
    plt.tight_layout()
    plt.show()